<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-02-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 18 - Notebook: Grafos de Logística de Insumos e Produtos Acabados na Linha de Envase

Neste notebook implementamos a modelagem em rede multicamada da **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)**: um Grafo Acíclico Dirigido (DAG) para o fluxo físico de insumos, envase e paletização ($G_M$) e um dígrafo em anel para a circulação interna de caminhões ($G_V$).

In [ ]:
from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Tuple, Any


def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str


class GrafoLogistico:
    def __init__(self) -> None:
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError("Dijkstra exige pesos não negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError("Origem e destino devem pertencer ao grafo.")

        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f"Não existe rota de {origem} para {destino}.")

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho


def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)


# ------------------------------------------------------------------------------
# 1. GM: Fluxo de Materiais na Linha de Envase de Bebidas (DAG)
# ------------------------------------------------------------------------------
fluxo_materiais = GrafoLogistico()
adicionar_rotas(fluxo_materiais, [
    ("Portaria", "Balança de Entrada", 25.0, "liberação e pesagem do recebimento"),
    ("Balança de Entrada", "Recebimento / Galpão A", 55.0, "direcionamento à descarga de insumos"),

    # Recebimento de matérias-primas e embalagens no Galpão A
    ("Recebimento / Galpão A", "Tanque: Xarope Concentrado", 18.0, "armazenagem de base de xarope"),
    ("Recebimento / Galpão A", "Reservatório: Água Tratada", 22.0, "armazenagem de água filtrada"),
    ("Recebimento / Galpão A", "Bateria: CO2 Líquido", 26.0, "recebimento de gás para carbonatação"),
    ("Recebimento / Galpão A", "Almoxarifado: Garrafas PET", 30.0, "armazenagem de vasilhames vazios"),
    ("Recebimento / Galpão A", "Almoxarifado: Tampas e Rótulos", 32.0, "armazenagem de insumos de fechamento"),

    # Preparação e alimentação da malha de envase (SCADA-Core)
    ("Tanque: Xarope Concentrado", "Misturador / Xaroparia", 25.0, "transferência por bombeamento sanitário"),
    ("Reservatório: Água Tratada", "Misturador / Xaroparia", 24.0, "linha de diluição padronizada"),
    ("Misturador / Xaroparia", "TS1_Suprimento", 30.0, "alimentação do tanque principal de bebida"),
    ("Bateria: CO2 Líquido", "TS1_Suprimento", 45.0, "injeção pressurizada de carbonatação"),

    # Malha hidráulica da Linha de Envase (Aulas 11 a 15)
    ("TS1_Suprimento", "VS1 / BC1_Bomba / AS1", 18.0, "sucção, recalque e acumulação hidráulica"),
    ("VS1 / BC1_Bomba / AS1", "EST_Envase", 10.0, "dosagem via VS2 e SQ2 no bico injetor"),

    # Alimentação de garrafas e tampas na esteira RC1
    ("Almoxarifado: Garrafas PET", "Lavadora / Rinser de Garrafas", 40.0, "transporte para enxágue sanitário"),
    ("Lavadora / Rinser de Garrafas", "EST_Envase", 35.0, "entrada de garrafas limpas na esteira RC1"),
    ("Almoxarifado: Tampas e Rótulos", "EST_Envase", 75.0, "alimentação do cabeçote rosqueador"),

    # Pós-envase, paletização e estoque no Galpão B
    ("EST_Envase", "Rotuladora e Enfardadeira", 22.0, "aplicação de rótulo e termoencolhimento"),
    ("Rotuladora e Enfardadeira", "Paletização Automatizada", 28.0, "formação de paletes padrão PBR"),
    ("Paletização Automatizada", "Estoque: Refrigerante Cola", 30.0, "endereçamento de lote acabado"),
    ("Paletização Automatizada", "Estoque: Refrigerante Guaraná", 35.0, "endereçamento de lote acabado"),
    ("Paletização Automatizada", "Estoque: Água Gaseificada", 40.0, "endereçamento de lote acabado"),

    # Expedição final nas docas
    ("Estoque: Refrigerante Cola", "Docas de Expedição", 22.0, "separação para embarque rodoviário"),
    ("Estoque: Refrigerante Guaraná", "Docas de Expedição", 25.0, "separação para embarque rodoviário"),
    ("Estoque: Água Gaseificada", "Docas de Expedição", 28.0, "separação para embarque rodoviário"),
])

dist_total, caminho_total = fluxo_materiais.menor_caminho("Portaria", "Docas de Expedição")
print("=== FLUXO DE MATERIAIS ATÉ A EXPEDIÇÃO (GM - LINHA DE ENVASE) ===")
print("  " + " -> ".join(caminho_total))
print(f"  Distância didática total: {dist_total:.1f} m\n")

assert "TS1_Suprimento" in caminho_total
assert "EST_Envase" in caminho_total
assert "Paletização Automatizada" in caminho_total
assert dist_total == 281.0, f"Distância esperada de 281.0 m, obtido: {dist_total}"

insumos = [
    "Tanque: Xarope Concentrado",
    "Reservatório: Água Tratada",
    "Bateria: CO2 Líquido",
    "Almoxarifado: Garrafas PET",
    "Almoxarifado: Tampas e Rótulos",
]

tabela_insumos = []
for insumo in insumos:
    d_ins, c_ins = fluxo_materiais.menor_caminho(insumo, "Docas de Expedição")
    tabela_insumos.append({
        "Insumo da Linha de Envase": insumo,
        "Distância até Doca (m)": f"{d_ins:.1f}",
        "Rota Completa no Processo": " -> ".join(c_ins)
    })

print("=== ROTAS DOS INSUMOS ATÉ AS DOCAS DE EXPEDIÇÃO ===")
print(formatar_tabela(tabela_insumos))


# ------------------------------------------------------------------------------
# 2. GV: Circulação Interna de Veículos (Anel Viário da Planta)
# ------------------------------------------------------------------------------
circulacao_veiculos = GrafoLogistico()
anel_viario = [
    ("Portaria", "Balança de Entrada", 25.0, "entrada de caminhões"),
    ("Balança de Entrada", "Pátio de Recebimento", 55.0, "acesso às docas de descarga"),
    ("Pátio de Recebimento", "Galpão A (Insumos)", 40.0, "rota de descarga de matérias-primas"),
    ("Galpão A (Insumos)", "Área de Processo e Envase", 65.0, "via interna de serviço"),
    ("Área de Processo e Envase", "Galpão B (Acabados)", 75.0, "acesso ao armazém de paletes"),
    ("Galpão B (Acabados)", "Docas de Expedição", 30.0, "posicionamento para carregamento"),
    ("Docas de Expedição", "Balança de Saída", 60.0, "pesagem de saída de caminhões"),
    ("Balança de Saída", "Portaria", 35.0, "liberação fiscal e saída"),
]
adicionar_rotas(circulacao_veiculos, anel_viario)

distancia_saida, rota_saida = circulacao_veiculos.menor_caminho(
    "Docas de Expedição", "Portaria"
)
print("\n=== CIRCULAÇÃO DE VEÍCULO APÓS O CARREGAMENTO (GV) ===")
print("  " + " -> ".join(rota_saida))
print(f"  Distância didática total: {distancia_saida:.1f} m")

assert rota_saida == ["Docas de Expedição", "Balança de Saída", "Portaria"]
assert distancia_saida == 95.0
print("\n✓ Modelos logísticos (GM e GV) da Linha de Envase validados com 100% de sucesso!")